# AttackDRO — 8/255 anchor rows on Colab (avg_frozen + 3a, 3 seeds)Fills two paper-table **in-house anchor rows** at the locked protocol eps=(8/255, 0.5, 12), train==eval:- **avg_frozen** (`configs/avg_frozen.yaml`) — uniform q (F4 anchor).- **3a = bindaware** (`configs/bindaware.yaml`, `weight_signal=robust_acc`) — signal-axis row.**Why Colab is fine here:** these are pure training + APGD eval; no hardware-specific FLOPsmeasurement (that's only CARD-PB, which stays on the 5070ti). Do **not** run predictive/CARD-PBor the R1/R3/R2 floor fixes here.**Pinned to commit `b73d078`** on branch `card-pb-fixes` — the snapshot with the 8/255 relockand these configs. The later R1/R3/R2 fixes touch only the `predictive_binding` path, so thispin gives byte-identical avg_frozen/3a behavior and a checkpoint format identical to local.**Disconnect-proof:** Drive-mounted; dataset + checkpoints + eval JSONs live on Drive; per-seedskip-if-done and checkpoint-restore guards → re-running the notebook resumes exactly where itstopped. W&B **online**, project `attackdro-union`, same schema as local.**Secrets:** set Colab userdata `GH_PAT` (repo read) and `WANDB_API_KEY`, or paste when prompted.

In [ ]:
# 1) Mount Drive + persistent paths (survive disconnects)from google.colab import drive; drive.mount('/content/drive')import osDRIVE = '/content/drive/MyDrive/attackdro'for sub in ('results','checkpoints','data'):    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)print('Drive workspace ready:', DRIVE)

In [ ]:
# 2) Clone the repo and pin the exact reviewed snapshot (b73d078)import os, subprocessSHA  = 'b73d078'                      # card-pb-fixes pre-fix snapshot (8/255 + avg_frozen/bindaware)REPO = 'github.com/anhkiet287/attackdro.git'try:    from google.colab import userdata; PAT = userdata.get('GH_PAT')except Exception:    PAT = Noneif not PAT:    from getpass import getpass; PAT = getpass('GitHub PAT (repo read): ')url = f'https://{PAT}@{REPO}'if not os.path.isdir('/content/attackdro/.git'):    subprocess.run(['git','clone','-q',url,'/content/attackdro'], check=True)os.chdir('/content/attackdro')subprocess.run(['git','fetch','-q','origin','card-pb-fixes'], check=True)subprocess.run(['git','checkout','-q',SHA], check=True)head = subprocess.run(['git','rev-parse','--short','HEAD'], capture_output=True, text=True).stdout.strip()assert head.startswith(SHA[:7]), f'HEAD {head} != {SHA}'print('repo @', head, '(pinned)')

In [ ]:
# 3) Deps (torch/torchvision already on Colab; match requirements.txt for the rest)!pip -q install numpy pyyaml tqdm matplotlib wandb==0.28.0!pip -q install "autoattack @ git+https://github.com/fra31/auto-attack.git"print('deps installed')

In [ ]:
# 4) Path + EPS GUARD — refuse to run on the wrong protocolimport os, sysos.environ['PYTHONPATH'] = 'src'          # scripts also self-insert, belt-and-suspenderssys.path.insert(0, 'src')from robustdro.utils.io import load_configb = load_config('configs/base.yaml')['threat_model']eps = b['linf']['eps']assert abs(eps - 8/255) < 1e-9, f'WRONG eps {eps} (expected 8/255) — wrong branch/SHA, STOP'assert b['l2']['eps'] == 0.5 and b['l1']['eps'] == 12.0, f'l2/l1 mismatch: {b}'print(f'EPS GUARD OK: linf={eps:.8f} (8/255), l2=0.5, l1=12  ==  local base.yaml')

In [ ]:
# 5) W&B online (project attackdro-union, entity=default; same schema as local)import osos.environ['WANDB_MODE'] = 'online'try:    from google.colab import userdata; K = userdata.get('WANDB_API_KEY')except Exception:    K = Noneif not K:    from getpass import getpass; K = getpass('W&B API key: ')os.environ['WANDB_API_KEY'] = Kimport wandb; wandb.login(key=K)print('W&B online -> attackdro-union (entity = API-key default)')

In [ ]:
# 6) SMOKE both configs first (2 steps, wandb disabled) — proves data->model->backward->eval path.#    Also triggers the one-time CIFAR-10 download into the Drive cache.import subprocess, osDATA = f'{DRIVE}/data'env = dict(os.environ, PYTHONPATH='src', WANDB_MODE='disabled')for cfg in ('configs/avg_frozen.yaml','configs/bindaware.yaml'):    print('=== SMOKE', cfg, '===', flush=True)    subprocess.run(['python','scripts/train.py','--config',cfg,'--smoke',                    '--wandb-mode','disabled','--set',f'dataset.root={DATA}'],                   check=True, env=env)print('SMOKE OK — both anchor configs run end-to-end; CIFAR cached on Drive')

In [ ]:
# 7) Train + eval the anchor rows, 3 seeds each. Disconnect-proof:#    - skip a seed whose eval JSON is already on Drive#    - if its best checkpoint is on Drive but eval isn't, restore + eval (don't retrain)#    - copy checkpoint + eval JSON to Drive immediately per unitimport subprocess, os, shutil, globDATA = f'{DRIVE}/data'env = dict(os.environ, PYTHONPATH='src', WANDB_MODE='online')JOBS = [('configs/avg_frozen.yaml','avg_frozen_8255','F4 uniform anchor'),        ('configs/bindaware.yaml','bindaware_8255','3a signal-axis (weight_signal=robust_acc)')]for cfg, tag, note in JOBS:    for S in (0,1,2):        run = f'{tag}_s{S}'        drive_eval = f'{DRIVE}/results/eval_{run}.json'        best_drive = f'{DRIVE}/checkpoints/{run}_best.pt'        if os.path.exists(drive_eval):            print('SKIP (done):', run); continue        os.makedirs('checkpoints', exist_ok=True)        if os.path.exists(best_drive):            shutil.copy(best_drive, f'checkpoints/{run}_best.pt')            print('RESTORED checkpoint, skipping train:', run, flush=True)        else:            print(f'=== TRAIN {run}  ({note}) ===', flush=True)            subprocess.run(['python','scripts/train.py','--config',cfg,'--seed',str(S),                            '--run-name',run,'--wandb-mode','online',                            '--set',f'dataset.root={DATA}'], check=True, env=env)            for ck in glob.glob(f'checkpoints/{run}_*.pt'):                shutil.copy(ck, f'{DRIVE}/checkpoints/'+os.path.basename(ck))        print(f'=== EVAL {run} (APGD n=1000) ===', flush=True)        subprocess.run(['python','scripts/evaluate.py','--config',cfg,                        '--checkpoint',f'checkpoints/{run}_best.pt','-n','1000','--version','apgd',                        '--run-name',run,'--tier','in-house','--out',f'results/eval_{run}.json',                        '--set',f'dataset.root={DATA}'], check=True, env=env)        shutil.copy(f'results/eval_{run}.json', drive_eval)        print('DONE', run, '-> Drive + W&B', flush=True)print('ALL ANCHOR RUNS COMPLETE')

In [ ]:
# 8) 3-seed table (reads the Drive JSONs — reflects current progress even mid-run)import json, os, statistics as stfor tag in ('avg_frozen_8255','bindaware_8255'):    rows = []    for S in (0,1,2):        f = f'{DRIVE}/results/eval_{tag}_s{S}.json'        if os.path.exists(f):            rows.append((S, json.load(open(f))['metrics']))    if not rows:        print(f'{tag}: no results yet'); continue    u = [100*m['worst_union_acc'] for _, m in rows]    band = f'±{st.pstdev(u):.2f}' if len(u) > 1 else ''    seeds = ', '.join(f's{S}={100*m["worst_union_acc"]:.1f}' for S, m in rows)    print(f'\n{tag}: UNION {st.mean(u):.1f}{band}  (n={len(u)})  [{seeds}]  vs RAMP 46.1')    for S, m in rows:        pn = m['per_norm_robust_acc']        print(f'  s{S}: union {100*m["worst_union_acc"]:.1f}  clean {100*m["clean_acc"]:.1f}  '              f'linf {100*pn["linf"]:.1f}  l2 {100*pn["l2"]:.1f}  l1 {100*pn["l1"]:.1f}')